# Доверительные интервалы для коэффициентов

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col # вывод результатов тестирования
import matplotlib.pyplot as plt

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

<div style="background-color:Bisque; color:DarkBlue; padding:30px;">

<i><b><span style="color: purple">Доверительные интервалы для коэффициентов</span></b></i><br>

Пусть $\gamma$ — доверительная вероятность (обычно 90%, 95%, 99%). Ей соответствует уровень значимости $\alpha=1-\gamma$

Доверительный интервал для коэффициента $\beta$

$$
\mathcal{I}_\gamma=\left(\hat{\beta}-s.e.(\beta)\cdot t_{cr}\; ;\;\hat{\beta}+s.e.(\beta)\cdot t_{cr}\right)
$$

где $t_{cr}=t_{df=n-k-1}(\alpha)$

__Интерпретация:__ $P(\beta\in\mathcal{I}_\gamma)=\gamma$.

__Замечание:__

* доверительный интервал можно рассматривать как "точность оценивания" коэффициента
* в зависимости от используемых $s.e.(\beta)$ __доверительный интервал м.б. неробастным и робастным__
* при выводе протокола подгонки модели методом `.summary(slim=False, alpha=0.05)` границы доверительных интервалов с $\gamma=1-\alpha$ для коэффициентов — это два последних столбца в таблице с коэффициентами
* границы доверительных интервалов для коэффициентов можно получить вызвав метод `.conf_int(alpha=0.05)`

**<span style="color:purple">Доверительный интервал и t-тест:</span>** доверительный интервал можно использовать для тестирования гипотезы

$$
\begin{aligned}
H_0:\beta&=\theta & &vs & H_1:\beta&\ne\theta
\end{aligned}
$$

__Вывод__: при уровне значимости $\alpha$ не отвергаем гипотезу $\iff \theta\in \mathcal{I}_\gamma$. В частности, коэффициент незначим $\iff 0\in \mathcal{I}_\gamma$

</div>

Для набора данных `sleep75` рассмотрим регрессию

$$
sleep=\beta_0+\beta_1 totwrk+\beta_2 age+\beta_3 age^2+\beta_4 smsa+\beta_5 male+\beta_6 south+\beta_7 marr+\beta_8 union+error
$$

In [ ]:
# импорт данных
df = pd.read_csv('../datasets/sleep75.csv')

## Неробастный доверительный интервал

Подгоним модель и выведем протокол подгонки с `90%` доверительными интервалами для коэффициентов

In [ ]:
# спецификация модели
mod = ols(formula='sleep~1+totwrk+age+I(age**2)+smsa+male+south+marr+union', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res_ols = mod.fit()

In [ ]:
# протокол подгонки с 90% доверительными интервалами
res_ols.summary(slim=True, alpha=0.1)

In [ ]:
# неробастные доверительные интервалы
res_ols.conf_int(alpha=0.1).round(3)

**ВЫВОД**: коэффициент значим на уровне 10%, если 90%-доверительный интервал не содержит ноль. По неробастным стандартным ошибкам на уровне значимости 10% значимы коэффициенты при `totwrk`, `male` и `south`; интервалы для коэффициентов при `age`, `I(age ** 2)`, `smsa`, `marr` и `union` содержат ноль, эти коэффициенты незначимы.

## Робастный доверительный интервал

Подгоним модель и выведем протокол подгонки с `99%` доверительными интервалами для коэффициентов

In [ ]:
# подгонка модели с робастной HC3-оценкой ковариационной матрицы
res_hc = mod.fit(cov_type='HC3', use_t=True)

In [ ]:
# протокол подгонки с 99% доверительными интервалами
res_hc.summary(slim=True, alpha=0.01)

In [ ]:
# робастные доверительные интервалы
res_hc.conf_int(alpha=0.01).round(3)

**ВЫВОД**: коэффициент значим на уровне 1%, если 99%-доверительный интервал не содержит ноль. По робастным стандартным ошибкам (HC3) на уровне значимости 1% значим только коэффициент при `totwrk`. Набор значимых коэффициентов стал уже, чем по неробастным 90%-интервалам (интервалы для `male` и `south` теперь содержат ноль), но причина — более строгий уровень значимости, а не робастные стандартные ошибки: на одном и том же уровне (и 10%, и 1%) неробастные и робастные интервалы дают один и тот же набор значимых коэффициентов.

## Визуализация доверительных интервалов

In [ ]:
# неробастные 90% доверительные интервалы
conf_int = res_ols.conf_int(alpha=0.1)
# палитра цветов для интервалов
colors = plt.cm.Dark2(np.linspace(0, 1, len(mod.exog_names)))
# построение графика (без константы)
plt.figure()
for i in range(1, len(mod.exog_names)):
    # оценку коэффициента отмечаем точкой
    plt.plot(res_ols.params.iloc[i], i, marker='o', markersize=5, color=colors[i])
    # соединяем границы интервала
    plt.hlines(y=i, xmin=conf_int.iloc[i, 0], xmax=conf_int.iloc[i, 1], color=colors[i])
# подписи на оси y
plt.yticks(range(1, len(mod.exog_names)), mod.exog_names[1:], fontstyle='italic', family='serif')
# цвета подписей оси y из списка цветов colors
for label, color in zip(plt.gca().get_yticklabels(), colors[1:]):
    label.set_color(color)
plt.xlabel('Оценка параметра', fontsize=14, fontstyle='italic', family='serif')
plt.ylabel('Регрессор', fontsize=14, fontstyle='italic', family='serif')
plt.title('90% доверительные интервалы для коэффициентов', fontsize=18, fontweight='bold')
plt.grid()
# оптимизируем поля и расположение объектов
plt.tight_layout()
plt.show()

**ВЫВОД**: на рисунке 90%-доверительные интервалы для коэффициентов при `age`, `smsa`, `marr` и `union` пересекают ноль (для `smsa` — едва: правая граница равна 1.063), интервалы для `male` и `south` лежат правее нуля. Коэффициенты имеют разный масштаб, поэтому узкие интервалы (при `totwrk` и `I(age ** 2)`) на общей оси выглядят точкой, и их положение относительно нуля видно только по таблице: интервал для `totwrk` ноль не содержит, для `I(age ** 2)` — содержит.

## Доверительные интервалы и сравнение моделей

Оценим регрессию `sleep` на `totwrk`, `age`, `age**2`, `smsa`, `male`, `marr`, `union` по трём выборкам: Север (`south = 0`), Юг (`south = 1`) и общая — и сравним 90%-доверительные интервалы для коэффициентов (неробастные стандартные ошибки).

In [ ]:
# специфицируем и подгоним модель на разных подвыборках с неробастной ковариационной матрицей
mod_overall = ols(formula='sleep~1+totwrk+age+I(age**2)+smsa+male+marr+union', data=df)
res_overall = mod_overall.fit()
# север: south=0
mod_north = ols(formula='sleep~1+totwrk+age+I(age**2)+smsa+male+marr+union', data=df, subset=(df['south']==0))
res_north = mod_north.fit()
# юг: south=1
mod_south = ols(formula='sleep~1+totwrk+age+I(age**2)+smsa+male+marr+union', data=df, subset=(df['south']==1))
res_south = mod_south.fit()

In [ ]:
# результаты трёх оцениваний в одной таблице
summary_col([res_north, res_south, res_overall], model_names=['Север', 'Юг', 'Общая'], stars=True)

In [ ]:
# создадим объекты класса figure и axes
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3, sharex=False, sharey=True, figsize=(10, 4.5))
# общий заголовок
fig.suptitle('90% доверительные интервалы для коэффициентов', fontsize=18, fontweight='bold')
# общие подписи осей
fig.supxlabel('Оценка параметра', fontsize=14, fontstyle='italic', family='serif')
fig.supylabel('Регрессор', fontsize=14, fontstyle='italic', family='serif')

# Север: оценки коэффициентов — точки, доверительные интервалы — отрезки
conf_int = res_north.conf_int(alpha=0.1)
for i in range(1, len(mod_overall.exog_names)):
    ax1.plot(res_north.params.iloc[i], i, marker='o', markersize=5, color='lightseagreen')
    ax1.hlines(y=i, xmin=conf_int.iloc[i, 0], xmax=conf_int.iloc[i, 1], color='lightseagreen')
ax1.set_title('Север')
ax1.grid()

# Юг: оценки коэффициентов — точки, доверительные интервалы — отрезки
conf_int = res_south.conf_int(alpha=0.1)
for i in range(1, len(mod_overall.exog_names)):
    ax2.plot(res_south.params.iloc[i], i, marker='o', markersize=5, color='royalblue')
    ax2.hlines(y=i, xmin=conf_int.iloc[i, 0], xmax=conf_int.iloc[i, 1], color='royalblue')
ax2.set_title('Юг')
ax2.grid()

# Общая: оценки коэффициентов — точки, доверительные интервалы — отрезки
conf_int = res_overall.conf_int(alpha=0.1)
for i in range(1, len(mod_overall.exog_names)):
    ax3.plot(res_overall.params.iloc[i], i, marker='o', markersize=5, color='crimson')
    ax3.hlines(y=i, xmin=conf_int.iloc[i, 0], xmax=conf_int.iloc[i, 1], color='crimson')
ax3.set_title('Общая')
ax3.grid()

# подписи на оси y
plt.yticks(range(1, len(mod_overall.exog_names)), mod_overall.exog_names[1:])
# оптимизируем поля и расположение объектов
plt.tight_layout()
plt.show()

In [ ]:
# 90% доверительные интервалы для трёх оцениваний
conf_int_north = res_north.conf_int(alpha=0.1)
conf_int_south = res_south.conf_int(alpha=0.1)
conf_int_overall = res_overall.conf_int(alpha=0.1)
# построение графика: оценки коэффициентов — точки, доверительные интервалы — отрезки
plt.figure()
for i in range(1, len(mod_overall.exog_names)):
    plt.plot(res_north.params.iloc[i], i+0.2, marker='o', markersize=5, color='lightseagreen')
    line1 = plt.hlines(y=i+0.2, xmin=conf_int_north.iloc[i, 0], xmax=conf_int_north.iloc[i, 1], color='lightseagreen')
    plt.plot(res_south.params.iloc[i], i, marker='o', markersize=5, color='royalblue')
    line2 = plt.hlines(y=i, xmin=conf_int_south.iloc[i, 0], xmax=conf_int_south.iloc[i, 1], color='royalblue')
    plt.plot(res_overall.params.iloc[i], i-0.2, marker='o', markersize=5, color='crimson')
    line3 = plt.hlines(y=i-0.2, xmin=conf_int_overall.iloc[i, 0], xmax=conf_int_overall.iloc[i, 1], color='crimson')
# подписи на оси y
plt.yticks(range(1, len(mod_overall.exog_names)), mod_overall.exog_names[1:], fontstyle='italic', family='serif')
plt.xlabel('Оценка параметра', fontsize=14, fontstyle='italic', family='serif')
plt.ylabel('Регрессор', fontsize=14, fontstyle='italic', family='serif')
plt.title('90% доверительные интервалы для коэффициентов', fontsize=18, fontweight='bold')
plt.legend([line1, line2, line3], ['Север', 'Юг', 'Общая'])
plt.grid()
# оптимизируем поля и расположение объектов
plt.tight_layout()
plt.show()

**ВЫВОД**: 90%-доверительные интервалы моделей «Север» и «Юг» пересекаются для всех коэффициентов; интервалы для Юга шире (130 наблюдений против 576 для Севера). Пересечение интервалов не является тестом равенства коэффициентов; гипотеза об отсутствии структурных сдвигов проверяется F-тестом (тест Чоу), см. разбор `f-test-structural-breaks.ipynb`.